<a href="https://colab.research.google.com/github/Ymin-2/ESAA/blob/main/0306_%EC%84%B8%EC%85%98_%EB%AA%A8%EB%8D%B8%ED%9B%88%EB%A0%A8_%EC%97%B0%EC%8A%B5%EB%AC%B8%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **모델 훈련 연습 문제**
___
- 출처 : 핸즈온 머신러닝 Ch04 연습문제 1, 5, 9, 10
- 개념 문제의 경우 텍스트 셀을 추가하여 정답을 적어주세요.

### **1. 수백만 개의 특성을 가진 훈련 세트에서는 어떤 선형 회귀 알고리즘을 사용할 수 있을까요?**
___


정규 방정식은 특성 수가 많으면 계산이 매우 느려진다

경사 하강법 기반 알고리즘 사용
- 배치 경사 하강법
- 확률적 경사 하강법
- 미니배치 경사 하강법

### **2. 배치 경사 하강법을 사용하고 에포크마다 검증 오차를 그래프로 나타내봤습니다. 검증 오차가 일정하게 상승되고 있다면 어떤 일이 일어나고 있는 걸까요? 이 문제를 어떻게 해결할 수 있나요?**
___

- 검증 오차가 최소가 되는 지점에서 학습 중단
- ridge, lasso, elasticnet 등 규제 적용

### **3. 릿지 회귀를 사용했을 때 훈련 오차가 검증 오차가 거의 비슷하고 둘 다 높았습니다. 이 모델에는 높은 편향이 문제인가요, 아니면 높은 분산이 문제인가요? 규제 하이퍼파라미터 $\alpha$를 증가시켜야 할까요 아니면 줄여야 할까요?**
___

높은 편향(high bias) 문제  
모델이 데이터의 패턴을 충분히 학습하지 못한 underfitting 상태  
--> 규제 강도를 줄여야 하므로 𝛼값을 감소시켜야 함

### **4. 다음과 같이 사용해야 하는 이유는?**
___
- 평범한 선형 회귀(즉, 아무런 규제가 없는 모델) 대신 릿지 회귀
- 릿지 회귀 대신 라쏘 회귀
- 라쏘 회귀 대신 엘라스틱넷

(1) 선형 회귀 대신 릿지 회귀  
→ 규제를 추가하여 과적합을 줄이고 일반화 성능을 높이기 위해

(2) 릿지 회귀 대신 라쏘 회귀  
→ L1 규제를 사용하여 불필요한 특성의 계수를 0으로 만들어 자동으로 feature selection을 하기 위해

(3) 라쏘 회귀 대신 엘라스틱넷  
→ L1과 L2 규제를 함께 사용하여 라쏘의 feature selection과 릿지의 안정성을 동시에 얻기 위해

### **추가) 조기 종료를 사용한 배치 경사 하강법으로 iris 데이터를 활용해 소프트맥스 회귀를 구현해보세요(사이킷런은 사용하지 마세요)**


---



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X = iris["data"][:, (2, 3)]   # petal length, petal width
y = iris["target"]

In [ ]:
# train / validation split
np.random.seed(2042)

test_ratio = 0.2
valid_ratio = 0.2
total_size = len(X)

test_size = int(total_size * test_ratio)
valid_size = int(total_size * valid_ratio)
train_size = total_size - test_size - valid_size

rnd_indices = np.random.permutation(total_size)

X_train = X[rnd_indices[:train_size]]
y_train = y[rnd_indices[:train_size]]

X_valid = X[rnd_indices[train_size:train_size + valid_size]]
y_valid = y[rnd_indices[train_size:train_size + valid_size]]

X_test = X[rnd_indices[train_size + valid_size:]]
y_test = y[rnd_indices[train_size + valid_size:]]

In [ ]:
# bias 추가
X_train = np.c_[np.ones((len(X_train), 1)), X_train]
X_valid = np.c_[np.ones((len(X_valid), 1)), X_valid]
X_test  = np.c_[np.ones((len(X_test), 1)), X_test]

In [ ]:
# one-hot encoding 함수 정의
def to_one_hot(y, n_classes=None):
    if n_classes is None:
        n_classes = np.max(y) + 1

    one_hot = np.zeros((len(y), n_classes))
    for i, label in enumerate(y):
        one_hot[i, label] = 1
    return one_hot

n_outputs = len(np.unique(y))

Y_train = to_one_hot(y_train, n_outputs)
Y_valid = to_one_hot(y_valid, n_outputs)
Y_test  = to_one_hot(y_test, n_outputs)

In [ ]:
# softmax 함수
def softmax(logits):
    logits_shifted = logits - np.max(logits, axis=1, keepdims=True)  # overflow 방지
    exp_values = np.exp(logits_shifted)
    return exp_values / np.sum(exp_values, axis=1, keepdims=True)

In [ ]:
# 손실 함수
# cross-entropy + L2 regularization
def compute_loss(X, Y, Theta, alpha=0.0, epsilon=1e-7):
    logits = X.dot(Theta)
    Y_proba = softmax(logits)

    cross_entropy = -np.mean(np.sum(Y * np.log(Y_proba + epsilon), axis=1))
    l2_penalty = 0.5 * np.sum(Theta[1:] ** 2)   # bias 제외

    return cross_entropy + alpha * l2_penalty

In [ ]:
# 예측 및 정확도 함수
def predict(X, Theta):
    logits = X.dot(Theta)
    Y_proba = softmax(logits)
    return np.argmax(Y_proba, axis=1)

def accuracy_score(X, y, Theta):
    y_pred = predict(X, Theta)
    return np.mean(y_pred == y)

In [ ]:
# 파라미터 초기화
n_inputs = X_train.shape[1]   # bias 포함 -> 3
np.random.seed(2042)
Theta = np.random.randn(n_inputs, n_outputs)

eta = 0.1
n_iterations = 5000
alpha = 0.01
epsilon = 1e-7

best_loss = np.inf
best_theta = None

patience = 20
patience_count = 0

m = len(X_train)

train_losses = []
valid_losses = []

In [ ]:
# Batch Gradient Descent + Early Stopping
for iteration in range(n_iterations):
    # (1) 예측 확률
    logits = X_train.dot(Theta)
    Y_proba = softmax(logits)

    # (2) gradient 계산
    error = Y_proba - Y_train
    gradients = (1 / m) * X_train.T.dot(error)

    # (3) L2 regularization (bias 제외)
    gradients[1:] += alpha * Theta[1:]

    # (4) 파라미터 업데이트
    Theta = Theta - eta * gradients

    # (5) 손실 기록
    train_loss = compute_loss(X_train, Y_train, Theta, alpha=alpha, epsilon=epsilon)
    valid_loss = compute_loss(X_valid, Y_valid, Theta, alpha=alpha, epsilon=epsilon)

    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    if iteration % 100 == 0:
        print(f"iteration: {iteration:4d} | train_loss: {train_loss:.4f} | valid_loss: {valid_loss:.4f}")

    # (6) Early Stopping
    if valid_loss < best_loss:
        best_loss = valid_loss
        best_theta = Theta.copy()
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= patience:
            print(f"\n조기 종료 발생: iteration {iteration}")
            break


# 가장 좋았던 파라미터 복원
Theta = best_theta

iteration:    0 | train_loss: 1.9267 | valid_loss: 1.9486
iteration:  100 | train_loss: 0.7381 | valid_loss: 0.7369
iteration:  200 | train_loss: 0.5834 | valid_loss: 0.5910
iteration:  300 | train_loss: 0.5132 | valid_loss: 0.5223
iteration:  400 | train_loss: 0.4716 | valid_loss: 0.4806
iteration:  500 | train_loss: 0.4428 | valid_loss: 0.4514
iteration:  600 | train_loss: 0.4210 | valid_loss: 0.4294
iteration:  700 | train_loss: 0.4035 | valid_loss: 0.4119
iteration:  800 | train_loss: 0.3889 | valid_loss: 0.3975
iteration:  900 | train_loss: 0.3764 | valid_loss: 0.3854
iteration: 1000 | train_loss: 0.3655 | valid_loss: 0.3751
iteration: 1100 | train_loss: 0.3559 | valid_loss: 0.3661
iteration: 1200 | train_loss: 0.3473 | valid_loss: 0.3582
iteration: 1300 | train_loss: 0.3396 | valid_loss: 0.3513
iteration: 1400 | train_loss: 0.3326 | valid_loss: 0.3451
iteration: 1500 | train_loss: 0.3263 | valid_loss: 0.3395
iteration: 1600 | train_loss: 0.3205 | valid_loss: 0.3346
iteration: 170

In [ ]:
# 최종 성능 평가
train_acc = accuracy_score(X_train, y_train, Theta)
valid_acc = accuracy_score(X_valid, y_valid, Theta)
test_acc = accuracy_score(X_test, y_test, Theta)

print("\n최종 결과")
print("Train accuracy :", train_acc)
print("Valid accuracy :", valid_acc)
print("Test accuracy  :", test_acc)


최종 결과
Train accuracy : 0.9666666666666667
Valid accuracy : 0.9666666666666667
Test accuracy  : 0.9333333333333333
